In [76]:
import os
import tempfile
import subprocess

import numpy as np
import matplotlib.pyplot as plt

In [77]:
def save_force_video(keypoints, forces, out_path, fps=30, scale=5, figsize=(6,6), dpi=100):
    keypoints = np.asarray(keypoints)
    forces = np.asarray(forces)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    x_min = np.nanmin(keypoints[:,:,0]) - 20
    x_max = np.nanmax(keypoints[:,:,0]) + 20
    y_min = np.nanmin(keypoints[:,:,1]) - 20
    y_max = np.nanmax(keypoints[:,:,1]) + 20

    skeleton = [
        (5,7),(7,9),
        (6,8),(8,10),
        (5,6),
        (5,11),(6,12),
        (11,12),
        (11,13),(13,15),
        (12,14),(14,16)
    ]

    with tempfile.TemporaryDirectory() as tmpdir:
        for frame_idx in range(keypoints.shape[0]):
            k = keypoints[frame_idx]
            f = forces[frame_idx]

            fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
            ax.scatter(k[:,0], k[:,1], c='red')
            for i, j in skeleton:
                ax.plot([k[i,0], k[j,0]], [k[i,1], k[j,1]], 'blue')

            if not np.any(np.isnan(k[15])):
                x, y = k[15]
                fx, fy, fz = f[0], f[1], f[2]
                ax.arrow(x, y, fx * scale, fy * scale, color='purple', head_width=5)

            if not np.any(np.isnan(k[16])):
                x, y = k[16]
                fx, fy, fz = f[3], f[4], f[5]
                ax.arrow(x, y, fx * scale, fy * scale, color='purple', head_width=5)

            ax.set_title(f'Frame {frame_idx}')
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_max, y_min)
            ax.set_aspect('equal')

            filename = os.path.join(tmpdir, f'frame_{frame_idx:04d}.png')
            fig.savefig(filename, dpi=dpi)
            plt.close(fig)

        cmd = [
            'ffmpeg',
            '-y',
            '-framerate', str(fps),
            '-i', os.path.join(tmpdir, 'frame_%04d.png'),
            '-c:v', 'libx264',
            '-pix_fmt', 'yuv420p',
            out_path,
        ]
        print('Running ffmpeg...')
        subprocess.run(cmd, check=True)

    print(f'Video saved to {out_path}')

In [78]:
kps = np.load("yolo_kps/keypoints_general_pose_1002.npy")
kps = kps[:kps.shape[0]-2, :, :]

forces = np.load("forceplate_arr/fp_041002.npy")
# forces = pred.squeeze(axis=1)

In [79]:
kps.shape, forces.shape

((383, 17, 2), (384, 6))

In [80]:
output_video = 'out_files/gen_pose_forceplate.mp4'
save_force_video(kps, forces, out_path=output_video, fps=30, scale=0.25)
print('Saved video to', output_video)

Running ffmpeg...


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

Video saved to out_files/gen_pose_forceplate.mp4
Saved video to out_files/gen_pose_forceplate.mp4


[out#0/mp4 @ 0x57cc99f48140] video:246kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 2.151321%
frame=  383 fps=0.0 q=-1.0 Lsize=     252kB time=00:00:12.66 bitrate= 162.8kbits/s speed=56.9x    
[libx264 @ 0x57cc99f48880] frame I:2     Avg QP:10.86  size:  8914
[libx264 @ 0x57cc99f48880] frame P:104   Avg QP:23.96  size:  1133
[libx264 @ 0x57cc99f48880] frame B:277   Avg QP:31.81  size:   419
[libx264 @ 0x57cc99f48880] consecutive B-frames:  1.3%  5.2%  4.7% 88.8%
[libx264 @ 0x57cc99f48880] mb I  I16..4: 80.2%  9.8% 10.0%
[libx264 @ 0x57cc99f48880] mb P  I16..4:  0.5%  0.9%  0.2%  P16..4:  3.1%  2.4%  1.1%  0.0%  0.0%    skip:91.9%
[libx264 @ 0x57cc99f48880] mb B  I16..4:  0.1%  0.1%  0.0%  B16..8:  5.5%  1.4%  0.2%  direct: 0.1%  skip:92.6%  L0:50.0% L1:45.2% BI: 4.8%
[libx264 @ 0x57cc99f48880] 8x8 transform intra:34.0% inter:22.7%
[libx264 @ 0x57cc99f48880] coded y,uvDC,uvAC intra: 7.7% 12.0% 10.6% inter: 0.5% 1.6% 1.4%
[libx264 @ 0x57cc99f48880] i16 v